# Lab 04 — Parquet na prática (DuckDB)

**Onde roda:** 🟢 Browser (JupyterLite).

Objetivo: **gravar e ler Parquet** (colunar, comprimido, com schema) e sentir por que ele vence o CSV no analítico.

In [ ]:
try:
    import duckdb
except ModuleNotFoundError:
    import piplite; await piplite.install('duckdb'); import duckdb
con = duckdb.connect()
con.execute('CREATE TABLE vendas(categoria VARCHAR, valor INTEGER, descricao VARCHAR)')
con.executemany('INSERT INTO vendas VALUES (?,?,?)', [
    ('A',100,'texto longo...'),('B',200,'texto longo...'),('A',50,'texto longo...')])
# Grava em PARQUET (colunar). O schema e os tipos vão embutidos.
con.execute("COPY vendas TO 'vendas.parquet' (FORMAT parquet)")
print('parquet gravado')

## 1. Ler o schema do Parquet
Diferente do CSV, o Parquet **carrega o schema** (nomes e tipos).

In [ ]:
con.execute("DESCRIBE SELECT * FROM 'vendas.parquet'").df()

## 2. Ler só as colunas necessárias (colunar)
O Parquet lê apenas `categoria` e `valor` — sem tocar na coluna 'gorda' `descricao`.

In [ ]:
con.execute('''
  SELECT categoria, SUM(valor) AS receita
  FROM 'vendas.parquet'
  GROUP BY categoria
  ORDER BY receita DESC
''').df()

## 3. Sua vez (mini-desafio)
Lendo do **Parquet**, traga a **receita total** (`SUM(valor)`) — um número. Verifique.

In [ ]:
resposta = con.execute("SELECT SUM(valor) FROM 'vendas.parquet'").fetchone()[0]
resposta

In [ ]:
def verificar(v):
    try:
        assert v == 350, 'A + A + B = 100 + 50 + 200 = 350.'
        print('✅ Correto! Parquet: colunar, comprimido e com schema — o formato do analítico.')
    except AssertionError as e:
        print('❌', e)

verificar(resposta)